In [ ]:
# ============================================================================
# IMPORTS - TODAS AS BIBLIOTECAS
# ============================================================================

# Bibliotecas padrão
import os
import pandas as pd
import numpy as np
import re
import unicodedata
import shutil

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder, LabelBinarizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils import shuffle

# ELI5 para análise de features
import eli5
from eli5.sklearn import PermutationImportance
from eli5 import show_weights, show_prediction

# TensorFlow/Keras para MLP
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scikeras.wrappers import KerasClassifier

# Transformers para BERT
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import torch

# Word2Vec (Gensim)
from gensim.models import Word2Vec

print("="*80)
print("IMPORTS CARREGADOS COM SUCESSO")
print("="*80)
print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
try:
    import gensim
    print(f"✅ Gensim: {gensim.__version__}")
except:
    print("⚠️  Gensim não encontrado. Instale com: pip install gensim")
print("="*80)


# CONFIGURAÇÕES

In [ ]:
BASE_FOLDER_TRAIN = "treino"

TRAIN_FILE = "ep2-train.csv"

preprocess_params = {
    # Normalização básica de texto
    "lowercase": True,             
    "normalize_unicode": False,      
    "remove_extra_whitespace": True, 
    "remove_punct": False,            
    
    # Normalização de entidades específicas
    "normalize_email": False,      # joao@email.com → <EMAIL>
    "normalize_url": False,        # http://site.com → <URL>
    "normalize_phone": False,      # (85) 98645-1524 → <PHONE>
    "normalize_date": False,       # 28/08/2018 → <DATE>
    "normalize_time": False,       # 14:30 → <TIME>
    "normalize_percent": False,    # 43.00% → <PERCENT>
    "normalize_document": False,   # CPF/CNPJ → <DOCUMENT>
    "normalize_code": False,       # RU101805325NL → <CODE>
    "normalize_law": False,        # Lei nº 12772/2012 → <LAW>
}

In [ ]:
# Configuração do Baseline (parâmetros fixos)
BASELINE_CONFIG = {
    'vectorizer': {
        'max_features': 10000,
        'ngram_range': (1, 2)
    },
    'model': {
        'C': 1.0,
        'solver': 'lbfgs',
        'class_weight': None,
        'max_iter': 1000,
        'random_state': 42
    }
}


# ANÁLISE DE BALANCEAMENTO DOS DATASETS


In [ ]:
def analisar_balanceamento(file_name):
    """Função para analisar o balanceamento de um dataset"""
    path = os.path.join(BASE_FOLDER_TRAIN, file_name)
    df = pd.read_csv(path, sep=";", encoding="latin1")
    
    print("="*60)
    print(f"ANÁLISE ESTATÍSTICA - {file_name}")
    print("="*60)
    
    # Informações básicas
    print(f"\n📊 INFORMAÇÕES GERAIS:")
    print(f"   • Total de linhas: {len(df):,}")
    print(f"   • Total de colunas: {len(df.columns)}")
    print(f"   • Colunas: {list(df.columns)}")
    
    # Verificar valores nulos
    print(f"\n🔍 VALORES NULOS:")
    print(f"   • Coluna 'req_text': {df['req_text'].isna().sum()}")
    print(f"   • Coluna 'profession': {df['profession'].isna().sum()}")
    
    # Distribuição das classes
    print(f"\n📈 DISTRIBUIÇÃO DAS CLASSES:")
    contagem_classes = df['profession'].value_counts()
    print(contagem_classes)
    
    print(f"\n📊 PORCENTAGEM POR CLASSE:")
    porcentagem_classes = df['profession'].value_counts(normalize=True) * 100
    for classe, perc in porcentagem_classes.items():
        count = contagem_classes[classe]
        print(f"   • {classe}: {count:,} ({perc:.2f}%)")
    
    # Verificar balanceamento
    print(f"\n⚖️ BALANCEAMENTO:")
    razao = contagem_classes.max() / contagem_classes.min()
    print(f"   • Razão maior/menor classe: {razao:.2f}x")
    if razao < 1.5:
        print(f"   • Status: ✅ Dataset bem balanceado")
    elif razao < 3:
        print(f"   • Status: ⚠️ Dataset moderadamente desbalanceado")
    else:
        print(f"   • Status: ❌ Dataset desbalanceado")
    
    print("\n" + "="*60)
    print()
    
    return df, contagem_classes


In [ ]:
resultados_analise = {}

df, contagem = analisar_balanceamento(TRAIN_FILE)
resultados_analise[TRAIN_FILE] = {
    'dataframe': df,
    'contagem_classes': contagem
}


# PRÉ-PROCESSAMENTO

In [ ]:
def normalize_entities(text, params):
    """
    Normaliza entidades específicas no texto, substituindo por tokens especiais.
    Ordem de aplicação é importante para evitar conflitos!
    """
    if not isinstance(text, str):
        return ""
    
    # 1. EMAIL - Captura endereços de email
    if params.get("normalize_email", False):
        text = re.sub(
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
            '<EMAIL>',
            text
        )
    
    # 2. URL - Links HTTP/HTTPS e www (MUITO MAIS ROBUSTO)
    if params.get("normalize_url", False):
        # URLs com protocolo (http, https, ftp) - SEM \b para evitar problemas
        # Captura até encontrar espaço, aspas, ou pontuação de fim de frase
        text = re.sub(
            r'(?:https?|ftp)://[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
        # URLs começando com www (incluindo www2, www3, etc)
        text = re.sub(
            r'www\d*\.[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
        # Domínios com caminho explícito
        text = re.sub(
            r'[a-z0-9][-a-z0-9]*\.[a-z]{2,}(?:\.[a-z]{2,})?/[^\s\)"<>]+',
            '<URL>',
            text,
            flags=re.IGNORECASE
        )
    
    # 3. PHONE - Telefones brasileiros
    if params.get("normalize_phone", False):
        # Formato: (85) 9 8645.1524, (85) 98645-1524, 85 98645-1524, etc
        text = re.sub(
            r'\(?\d{2}\)?\s?\d{4,5}[-.\s]?\d{4}',
            '<PHONE>',
            text
        )
    
    # 4. DATE - Várias formatações de data
    if params.get("normalize_date", False):
        # DD/MM/YYYY, DD-MM-YYYY, DD.MM.YYYY
        text = re.sub(
            r'\b\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}\b',
            '<DATE>',
            text
        )
        # YYYY/MM/DD, YYYY-MM-DD
        text = re.sub(
            r'\b\d{4}[/.-]\d{1,2}[/.-]\d{1,2}\b',
            '<DATE>',
            text
        )
    
    # 5. TIME - Horários (CORRIGIDO)
    if params.get("normalize_time", False):
        # Formato HH:MM ou HH:MM:SS
        text = re.sub(
            r'\b\d{1,2}:\d{2}(?::\d{2})?\b',
            '<TIME>',
            text
        )
        # Formato HHhMM (ex: 14h30)
        text = re.sub(
            r'\b\d{1,2}h\d{2}\b',
            '<TIME>',
            text
        )
    
    # 6. PERCENT - Porcentagens
    if params.get("normalize_percent", False):
        text = re.sub(
            r'\b\d+(?:[.,]\d+)?%',
            '<PERCENT>',
            text
        )
    
    # 7. DOCUMENT - CPF e CNPJ
    if params.get("normalize_document", False):
        # CPF: 123.456.789-00
        text = re.sub(
            r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b',
            '<DOCUMENT>',
            text
        )
        # CNPJ: 12.345.678/0001-00
        text = re.sub(
            r'\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b',
            '<DOCUMENT>',
            text
        )
    
    # 8. CODE - Códigos de rastreamento e processos
    if params.get("normalize_code", False):
        # Códigos de rastreamento (ex: RU101805325NL, PG326875631BR)
        text = re.sub(
            r'\b[A-Z]{2}\d{9,}[A-Z]{2}\b',
            '<CODE>',
            text
        )
        # Códigos de processo (ex: AC-2000-08012-000640, DAE 07.16.17365.94815-1)
        text = re.sub(
            r'\b[A-Z]{2,}-?\d{4}-\d{5}-\d{5,6}(?:-\d)?\b',
            '<CODE>',
            text
        )
        # DAE com pontos
        text = re.sub(
            r'\b\d{2}\.\d{2}\.\d{5}\.\d{5}-\d\b',
            '<CODE>',
            text
        )
    
    # 9. LAW - Referências legais
    if params.get("normalize_law", False):
        text = re.sub(
            r'\b(?:Lei|Portaria|Decreto|Resolução|Edital)\s+n[ºo°]?\s*\d+(?:/\d{4})?\b',
            '<LAW>',
            text,
            flags=re.IGNORECASE
        )
    
    return text


In [ ]:
def preprocess_operations(text, params):
    """
    Aplica operações de pré-processamento no texto.
    ORDEM IMPORTANTE: Normalização de entidades ANTES de lowercase e remoção de pontuação!
    
    Melhorias:
    - Validação de entrada mais robusta
    - Tratamento de casos especiais
    - Preservação de tokens especiais
    """
    # Validação de entrada
    if not isinstance(text, str):
        return ""
    
    if len(text.strip()) == 0:
        return ""
    
    # PASSO 1: Normalização de entidades (PRIMEIRO - antes de alterar case ou pontuação)
    text = normalize_entities(text, params)
    
    # PASSO 2: Normalização Unicode
    if params.get("normalize_unicode", False):
        text = unicodedata.normalize("NFKC", text)
    
    # PASSO 3: Lowercase
    if params.get("lowercase", False):
        text = text.lower()
    
    # PASSO 4: Remover pontuação (mas preservar tokens especiais <...>)
    if params.get("remove_punct", False):
        # Proteger tokens especiais temporariamente (padrão melhorado)
        text = re.sub(r'<(\w+)>', r'__TOKEN__\1__TOKEN__', text)
        # Remover pontuação (mas manter espaços e alfanuméricos)
        text = re.sub(r"[^\w\s]", " ", text)
        # Restaurar tokens especiais
        text = re.sub(r'__TOKEN__(\w+)__TOKEN__', r'<\1>', text)
    
    # PASSO 5: Remover espaços extras
    if params.get("remove_extra_whitespace", False):
        # Remover múltiplos espaços
        text = re.sub(r"\s+", " ", text)
        # Remover espaços no início e fim
        text = text.strip()
    
    # PASSO 6: Validação final
    if len(text.strip()) == 0:
        return ""
    
    return text

def preprocess_data(path, output_path, params):
    """
    Processa dados, salva no CSV e retorna dados preparados para treinamento.
    
    Melhorias:
    - Validação de dados mais robusta
    - Tratamento de erros
    - Logging de progresso
    - Validação de qualidade dos dados processados
    """
    col_text, col_label = "req_text", "profession"
    
    print("="*80)
    print("PRÉ-PROCESSAMENTO DE DADOS")
    print("="*80)
    
    # 1. Carregar dados originais
    if not os.path.exists(path):
        print(f"❌ Erro: {path} não encontrado.")
        return None
    
    print(f"\n📂 Carregando dados de: {path}")
    df = pd.read_csv(path, sep=";", encoding="latin1")
    print(f"   • Total de linhas carregadas: {len(df):,}")
    
    # 2. Validação inicial
    if col_text not in df.columns or col_label not in df.columns:
        print(f"❌ Erro: Colunas esperadas não encontradas.")
        print(f"   Colunas disponíveis: {list(df.columns)}")
        return None
    
    # 3. Limpeza inicial
    print(f"\n🧹 Limpando dados...")
    initial_count = len(df)
    
    # Remover linhas com valores nulos
    df = df[[col_text, col_label]].dropna()
    after_dropna = len(df)
    
    if initial_count != after_dropna:
        print(f"   • Removidas {initial_count - after_dropna} linhas com valores nulos")
    
    # Remover linhas com texto vazio
    df = df[df[col_text].astype(str).str.strip().str.len() > 0]
    after_empty = len(df)
    
    if after_dropna != after_empty:
        print(f"   • Removidas {after_dropna - after_empty} linhas com texto vazio")
    
    print(f"   • Total de linhas válidas: {len(df):,}")
    
    # 4. Aplicar pré-processamento
    print(f"\n⚙️  Aplicando pré-processamento...")
    print(f"   • Parâmetros ativos:")
    active_params = [k for k, v in params.items() if v]
    if active_params:
        for param in active_params:
            print(f"     - {param}")
    else:
        print(f"     (nenhum parâmetro de pré-processamento ativo)")
    
    # Processar com tratamento de erros
    processed_texts = []
    errors = 0
    
    for idx, text in enumerate(df[col_text]):
        try:
            processed = preprocess_operations(str(text), params)
            processed_texts.append(processed)
        except Exception as e:
            # Em caso de erro, usar texto vazio
            processed_texts.append("")
            errors += 1
        
        # Mostrar progresso a cada 10%
        if len(df) > 10 and (idx + 1) % (len(df) // 10) == 0:
            progress = ((idx + 1) / len(df)) * 100
            print(f"   • Progresso: {progress:.0f}%")
    
    if errors > 0:
        print(f"   ⚠️  Total de erros durante processamento: {errors}")
    
    df['req_text'] = processed_texts
    
    # 5. Validação pós-processamento
    print(f"\n✅ Validando dados processados...")
    
    # Remover linhas que ficaram vazias após processamento
    before_validation = len(df)
    df = df[df['req_text'].str.strip().str.len() > 0]
    after_validation = len(df)
    
    if before_validation != after_validation:
        print(f"   • Removidas {before_validation - after_validation} linhas vazias após processamento")
    
    # Estatísticas dos textos processados
    text_lengths = df['req_text'].str.len()
    print(f"   • Comprimento médio do texto: {text_lengths.mean():.1f} caracteres")
    print(f"   • Comprimento mínimo: {text_lengths.min()} caracteres")
    print(f"   • Comprimento máximo: {text_lengths.max()} caracteres")
    
    # 6. Manter apenas texto processado e profissão
    df_processed = df[['req_text', col_label]].copy()
    
    # 7. Salvar no CSV
    print(f"\n💾 Salvando dados processados...")
    df_processed.to_csv(output_path, sep=";", encoding="utf-8", index=False)
    print(f"   ✅ Arquivo salvo: {output_path}")
    print(f"   • Total de linhas salvas: {len(df_processed):,}")
    
    # 8. Preparar para treinamento
    print(f"\n🔄 Preparando dados para treinamento...")
    
    # Shuffle
    df_processed = shuffle(df_processed, random_state=10).reset_index(drop=True)
    
    # Label Encoding
    le = LabelEncoder()
    y = le.fit_transform(df_processed[col_label])
    
    # Mapeamento de labels (para referência)
    label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"   • Mapeamento de classes: {label_mapping}")
    
    # Distribuição de classes
    class_dist = pd.Series(y).value_counts().sort_index()
    print(f"   • Distribuição de classes:")
    for class_idx, class_name in enumerate(le.classes_):
        count = class_dist[class_idx]
        percentage = (count / len(y)) * 100
        print(f"     - {class_name}: {count:,} ({percentage:.2f}%)")
    
    # Features
    X = df_processed['req_text'].values
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=10
    )
    
    print(f"\n✅ Dados preparados para treinamento!")
    print(f"   • Treino: {len(X_train):,} textos")
    print(f"   • Teste: {len(X_test):,} textos")
    print(f"   • Proporção treino/teste: {len(X_train)/len(X_test):.2f}:1")
    
    # Estatísticas finais
    print(f"\n📊 Estatísticas finais:")
    print(f"   • Total de textos processados: {len(df_processed):,}")
    print(f"   • Total de classes: {len(le.classes_)}")
    print(f"   • Tamanho do conjunto de treino: {len(X_train):,}")
    print(f"   • Tamanho do conjunto de teste: {len(X_test):,}")
    
    return X_train, X_test, y_train, y_test, le

In [ ]:
# PROCESSAR DADOS, SALVAR NO CSV E PREPARAR PARA TREINAMENTO

input_file = os.path.join(BASE_FOLDER_TRAIN, TRAIN_FILE)
output_file = os.path.join(BASE_FOLDER_TRAIN, "ep2-train-preprocessed.csv")

print(f"\n📂 Arquivo de entrada: {input_file}")
print(f"📂 Arquivo de saída: {output_file}")

# Processar, salvar no CSV e preparar para treinamento (tudo em uma função)
result = preprocess_data(input_file, output_file, preprocess_params)

if result is not None:
    X_train, X_test, y_train, y_test, label_encoder = result
    
    # Armazenar para uso posterior
    dataset = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "label_encoder": label_encoder
    }
    
    print(f"\n{'='*100}")
    print(f"✅ PRÉ-PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
    print(f"{'='*100}")
    
    # Mostrar alguns exemplos
    print(f"\n📝 EXEMPLOS DE TEXTO PRÉ-PROCESSADO (primeiras 3 linhas de cada classe):")
    print("="*100)
    
    for class_idx, class_name in enumerate(label_encoder.classes_):
        class_indices = np.where(y_train == class_idx)[0][:3]
        if len(class_indices) > 0:
            print(f"\n{'='*60}")
            print(f"CLASSE: {class_name.upper()}")
            print(f"{'='*60}")
            
            for i, idx in enumerate(class_indices, 1):
                text = X_train[idx]
                print(f"\n  Exemplo {i}:")
                print(f"    {text[:200]}{'...' if len(text) > 200 else ''}")
                print(f"    (Comprimento: {len(text)} caracteres)")
else:
    print(f"\n{'='*100}")
    print(f"❌ ERRO NO PRÉ-PROCESSAMENTO")
    print(f"{'='*100}")
    dataset = {}

# TREINAMENTO

Classificação de textos entre autores **acadêmicos**, **privados** e **governamentais**.

In [ ]:
# ============================================================================
# BASELINE: TF-IDF + REGRESSÃO LOGÍSTICA
# ============================================================================

# Carregar dados
X_train = dataset["X_train"]
X_test = dataset["X_test"]
y_train = dataset["y_train"]
y_test = dataset["y_test"]
label_encoder = dataset["label_encoder"]

print("="*80)
print("BASELINE: TF-IDF + REGRESSÃO LOGÍSTICA")
print("="*80)

# 1. Criar Pipeline com parâmetros fixos
baseline_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(
        max_features=BASELINE_CONFIG['vectorizer']['max_features'],
        ngram_range=BASELINE_CONFIG['vectorizer']['ngram_range']
    )),
    ('model', LogisticRegression(
        C=BASELINE_CONFIG['model']['C'],
        solver=BASELINE_CONFIG['model']['solver'],
        class_weight=BASELINE_CONFIG['model']['class_weight'],
        max_iter=BASELINE_CONFIG['model']['max_iter'],
        random_state=BASELINE_CONFIG['model']['random_state']
    ))
])

# 2. Treinar modelo
print(f"\n📊 Parâmetros do Baseline:")
print(f"   TF-IDF: max_features={BASELINE_CONFIG['vectorizer']['max_features']}, "
      f"ngram_range={BASELINE_CONFIG['vectorizer']['ngram_range']}")
print(f"   Logistic Regression: C={BASELINE_CONFIG['model']['C']}, "
      f"solver={BASELINE_CONFIG['model']['solver']}, "
      f"class_weight={BASELINE_CONFIG['model']['class_weight']}")

print(f"\n🔄 Treinando modelo...")
baseline_pipeline.fit(X_train, y_train)
print(f"   ✅ Treinamento concluído!")

# 3. Validação Cruzada de 10 Folds
print(f"\n{'='*80}")
print("VALIDAÇÃO CRUZADA - 10 FOLDS")
print(f"{'='*80}")

print(f"\n🔄 Executando validação cruzada de 10 folds no conjunto de treino...")
cv_scores = cross_val_score(
    baseline_pipeline,
    X_train,
    y_train,
    cv=10,
    scoring='accuracy',
    n_jobs=-1
)

print(f"\n✅ Validação cruzada concluída!")
print(f"\n📊 Resultados da Validação Cruzada (10 folds):")
print(f"   • Acurácia média: {cv_scores.mean():.4f}")
print(f"   • Desvio padrão: {cv_scores.std():.4f}")
print(f"   • Acurácia mínima: {cv_scores.min():.4f}")
print(f"   • Acurácia máxima: {cv_scores.max():.4f}")
print(f"\n   Acurácias por fold:")
for fold_idx, score in enumerate(cv_scores, 1):
    print(f"     Fold {fold_idx}: {score:.4f}")

# 4. Teste Final com 15% separados
print(f"\n{'='*80}")
print("TESTE FINAL - 15% SEPARADOS")
print(f"{'='*80}")

baseline_test_score = baseline_pipeline.score(X_test, y_test)
baseline_y_pred = baseline_pipeline.predict(X_test)

print(f"\n📊 Resultados do Teste Final:")
print(f"   • Acurácia no conjunto de teste: {baseline_test_score:.4f}")
print(f"   • Total de amostras de teste: {len(y_test):,}")

# 5. Acurácia no conjunto de treino
baseline_train_score = baseline_pipeline.score(X_train, y_train)

print(f"\n{'='*80}")
print("RESULTADOS - BASELINE")
print(f"{'='*80}")
print(f"  Train Accuracy: {baseline_train_score:.4f}")
print(f"  CV Accuracy (10 folds): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Test Accuracy:  {baseline_test_score:.4f}")

# 6. Armazenar resultados
baseline_results = {
    'model_name': 'TF-IDF + Logistic Regression',
    'config': BASELINE_CONFIG,
    'train_score': baseline_train_score,
    'cv_scores': cv_scores,
    'cv_mean': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_score': baseline_test_score,
    'model': baseline_pipeline,
    'vectorizer': baseline_pipeline.named_steps['vectorizer'],
    'classifier': baseline_pipeline.named_steps['model'],
    'predictions': baseline_y_pred
}


In [ ]:
# ============================================================================
# ANÁLISE DE FEATURES IMPORTANTES - BASELINE (SIMPLIFICADA)
# ============================================================================

print("="*80)
print("ANÁLISE DE FEATURES IMPORTANTES - BASELINE")
print("="*80)

# Obter nomes das features (palavras/n-grams)
feature_names = baseline_results['vectorizer'].get_feature_names_out()

# Criar diretório para salvar figuras
figures_dir = "figuras"
os.makedirs(figures_dir, exist_ok=True)

# Análise de Top 10 Features por Classe
print("\n" + "-"*80)
print("TOP 10 FEATURES QUE FAVORECEM CADA CLASSE")
print("-"*80)

baseline_feature_analysis = {}

for class_idx, class_name in enumerate(label_encoder.classes_):
    print(f"\n{'='*60}")
    print(f"CLASSE: {class_name.upper()}")
    print(f"{'='*60}")
    
    # Obter coeficientes da classe
    coef = baseline_results['classifier'].coef_[class_idx]
    
    # Criar DataFrame com features e pesos
    features_df = pd.DataFrame({
        'feature': feature_names,
        'weight': coef
    })
    
    # Filtrar apenas features positivas (que favorecem a classe)
    positive_features = features_df[features_df['weight'] > 0].copy()
    
    # Ordenar por peso (maior primeiro)
    top_10 = positive_features.nlargest(10, 'weight')
    
    # Armazenar
    baseline_feature_analysis[class_name] = {
        'top_10_positive': top_10
    }
    
    # Mostrar resultados
    print(f"\nTop 10 Features que favorecem {class_name}:")
    print(top_10[['feature', 'weight']].to_string(index=False))
    
    # Criar e salvar figura
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(top_10)), top_10['weight'].values, color='steelblue')
    plt.yticks(range(len(top_10)), top_10['feature'].values, fontsize=10)
    plt.xlabel('Peso (Coeficiente)', fontsize=12)
    plt.title(f'Top 10 Features - {class_name.upper()}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    
    # Salvar figura
    figure_path = os.path.join(figures_dir, f'top10_features_{class_name}.png')
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ Figura salva: {figure_path}")
    plt.show()

# Matriz de Confusão
print(f"\n{'='*80}")
print("MATRIZ DE CONFUSÃO")
print(f"{'='*80}")

cm = confusion_matrix(y_test, baseline_y_pred)
cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.title('Matriz de Confusão - Baseline', fontsize=14, fontweight='bold')
plt.ylabel('Classe Verdadeira', fontsize=12)
plt.xlabel('Classe Predita', fontsize=12)
plt.tight_layout()

# Salvar matriz de confusão
confusion_path = os.path.join(figures_dir, 'confusion_matrix_baseline.png')
plt.savefig(confusion_path, dpi=300, bbox_inches='tight')
print(f"✅ Figura salva: {confusion_path}")
plt.show()

print("\nMatriz de Confusão:")
print(cm_df)

# Armazenar análise
baseline_results['feature_analysis'] = baseline_feature_analysis
baseline_results['confusion_matrix'] = cm_df

print(f"\n{'='*80}")
print("✅ ANÁLISE DE FEATURES CONCLUÍDA")
print(f"{'='*80}")


In [24]:
# ============================================================================
# IMPORTS PARA MLP E BERT
# ============================================================================

# TensorFlow/Keras para MLP
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scikeras.wrappers import KerasClassifier

# Transformers para BERT
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import torch
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import LabelBinarizer

# Word2Vec (Gensim)
from gensim.models import Word2Vec

# Configuração GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {'✅ Detectada' if torch.cuda.is_available() else '❌ Não detectada'}")

GPU: ❌ Não detectada


In [25]:
# ============================================================================
# FUNÇÕES WORD2VEC
# ============================================================================

def train_word2vec_on_data(X_train, vector_size=300, window=5, min_count=2, workers=4, sg=1, epochs=10):
    sentences = [texto.split() for texto in X_train]
    w2v_model = Word2Vec(sentences, vector_size=vector_size, window=window, 
                         min_count=min_count, workers=workers, sg=sg, epochs=epochs)
    return w2v_model

def text_to_vector(text, model, aggregation='mean'):
    words = text.split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    vectors = np.array(vectors)
    if aggregation == 'mean':
        return np.mean(vectors, axis=0)
    elif aggregation == 'sum':
        return np.sum(vectors, axis=0)
    elif aggregation == 'max':
        return np.max(vectors, axis=0)
    elif aggregation == 'min':
        return np.min(vectors, axis=0)
    else:
        raise ValueError(f"Método de agregação desconhecido: {aggregation}")

def texts_to_vectors(X, model, aggregation='mean', verbose=True):
    vectors = [text_to_vector(text, model, aggregation) for text in X]
    return np.array(vectors)

def prepare_data_with_word2vec(X_train, X_test, vector_size=300, window=5, min_count=2, aggregation='mean', verbose=True):
    w2v_model = train_word2vec_on_data(X_train, vector_size=vector_size, window=window, min_count=min_count)
    X_train_w2v = texts_to_vectors(X_train, w2v_model, aggregation=aggregation, verbose=verbose)
    X_test_w2v = texts_to_vectors(X_test, w2v_model, aggregation=aggregation, verbose=verbose)
    return X_train_w2v, X_test_w2v, w2v_model


In [ ]:
# ============================================================================
# CONFIGURAÇÃO MLP GRID-SEARCH
# ============================================================================

MLP_GRID_CONFIG = {
    'input_method': ['word2vec', 'tfidf'],  # 'tfidf', 'word2vec' ou ambos
    'tfidf_params': {
        'max_features': [10000],
        'ngram_range': [(1, 2)]
    },
    'word2vec_params': {
        'vector_size': [300],
        'window': [5],
        'min_count': [2],
        'aggregation': ['mean', 'sum']
    },
    'model_params': {
        'hidden_layers': [(512, 256)],
        'dropout_rate': [0.7],
        'learning_rate': [0.0001],
        'batch_size': [128]
    },
    'training_params': {
        'epochs': 30,
        'early_stopping_patience': 5,
        'validation_split': 0.15
    }
}


In [29]:
# ============================================================================
# MLP GRID-SEARCH SIMPLIFICADO (TF-IDF E WORD2VEC)
# ============================================================================

# Função para criar modelo MLP
def create_mlp_model(input_dim, hidden_layers, dropout_rate, learning_rate):
    """Cria modelo MLP com arquitetura configurável"""
    from keras.layers import Input
    
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    # Primeira camada oculta
    model.add(Dense(hidden_layers[0], activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    
    # Camadas ocultas adicionais
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    
    # Camada de saída (3 classes)
    model.add(Dense(3, activation='softmax'))
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Função para treinar e avaliar MLP completo: CV + Treino nos 85% + Teste nos 15%
def train_and_evaluate_mlp(X_train_prep, X_test_prep, y_train_oh, y_test_oh,
                            hidden_layers, dropout_rate, learning_rate, batch_size,
                            cv=3):
    """
    Treina e avalia MLP completo:
    1. CV nos 85% de treino (3 folds) - verbose=1
    2. Treino final nos 85% de treino - verbose=1
    3. Teste nos 15% separados
    
    Returns:
        cv_scores: Scores de validação cruzada
        test_score: Score no conjunto de teste (15%)
        model: Modelo treinado
    """
    from sklearn.model_selection import KFold
    
    # Converter para float32
    X_train_prep = X_train_prep.astype('float32')
    X_test_prep = X_test_prep.astype('float32')
    y_train_oh = y_train_oh.astype('float32')
    y_test_oh = y_test_oh.astype('float32')
    
    # Configurar modelo
    input_dim = X_train_prep.shape[1]
    
    def build_fn():
        return create_mlp_model(input_dim, hidden_layers, dropout_rate, learning_rate)
    
    # Callbacks customizados com prints para CV
    class EarlyStoppingWithPrint(EarlyStopping):
        def on_train_end(self, logs=None):
            if self.stopped_epoch > 0:
                print(f"  ⏹️  EarlyStopping ativado na época {self.stopped_epoch + 1}")
            super().on_train_end(logs)
    
    class ReduceLRWithPrint(ReduceLROnPlateau):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.last_lr = None
        
        def on_epoch_end(self, epoch, logs=None):
            if self.model is not None:
                current_lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
                if self.last_lr is None:
                    self.last_lr = current_lr
                super().on_epoch_end(epoch, logs)
                new_lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
                if self.last_lr != new_lr:
                    print(f"  📉 Learning rate reduzido: {self.last_lr:.2e} → {new_lr:.2e}")
                    self.last_lr = new_lr
    
    callbacks_cv = [
        EarlyStoppingWithPrint(monitor='loss', patience=5, restore_best_weights=True, verbose=0),
        ReduceLRWithPrint(monitor='loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
    ]
    
    # 1. VALIDAÇÃO CRUZADA nos 85% de treino
    kfold = KFold(n_splits=cv, shuffle=True, random_state=42)
    cv_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train_prep)):
        X_fold_train = X_train_prep[train_idx]
        y_fold_train = y_train_oh[train_idx]
        X_fold_val = X_train_prep[val_idx]
        y_fold_val = y_train_oh[val_idx]
        
        print(f"  📊 Fold {fold_idx + 1}/{cv} - Treinando...")
        
        fold_model = KerasClassifier(
            model=build_fn,
            batch_size=batch_size,
            epochs=MLP_GRID_CONFIG['training_params']['epochs'],
            verbose=1,  # Mostrar épocas
            callbacks=callbacks_cv,
            validation_split=0.0
        )
        
        fold_model.fit(X_fold_train, y_fold_train)
        fold_score = fold_model.score(X_fold_val, y_fold_val)
        cv_scores.append(fold_score)
        print(f"  ✅ Fold {fold_idx + 1}/{cv} - Score: {fold_score:.4f}")
    
    cv_scores = np.array(cv_scores)
    
    # Callbacks customizados com prints para treino final
    class EarlyStoppingWithPrint(EarlyStopping):
        def on_train_end(self, logs=None):
            if self.stopped_epoch > 0:
                print(f"  ⏹️  EarlyStopping ativado na época {self.stopped_epoch + 1}")
            super().on_train_end(logs)
    
    class ReduceLRWithPrint(ReduceLROnPlateau):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.last_lr = None
        
        def on_epoch_end(self, epoch, logs=None):
            if self.model is not None:
                current_lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
                if self.last_lr is None:
                    self.last_lr = current_lr
                super().on_epoch_end(epoch, logs)
                new_lr = float(keras.backend.get_value(self.model.optimizer.learning_rate))
                if self.last_lr != new_lr:
                    print(f"  📉 Learning rate reduzido: {self.last_lr:.2e} → {new_lr:.2e}")
                    self.last_lr = new_lr
    
    callbacks_final = [
        EarlyStoppingWithPrint(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
        ReduceLRWithPrint(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
    ]
    
    # 2. TREINO FINAL nos 85% de treino (completos)
    print(f"  📊 Treino Final - Treinando...")
    final_model = KerasClassifier(
        model=build_fn,
        batch_size=batch_size,
        epochs=MLP_GRID_CONFIG['training_params']['epochs'],
        verbose=1,  # Mostrar épocas
        callbacks=callbacks_final,
        validation_split=MLP_GRID_CONFIG['training_params']['validation_split']
    )
    
    final_model.fit(X_train_prep, y_train_oh)
    print(f"  ✅ Treino Final - Concluído")
    
    # 3. TESTE nos 15% separados
    test_score = final_model.score(X_test_prep, y_test_oh)
    
    return cv_scores, test_score, final_model


# ============================================================================
# GRID-SEARCH SIMPLIFICADO
# ============================================================================

# Carregar dados
X_train = dataset["X_train"]
X_test = dataset["X_test"]
y_train = dataset["y_train"]
y_test = dataset["y_test"]
label_encoder = dataset["label_encoder"]

# Label encoding (one-hot)
lb = LabelBinarizer()
y_train_oh = lb.fit_transform(y_train)
y_test_oh = lb.transform(y_test)

# Validar métodos configurados
methods_to_run = [m for m in ['tfidf', 'word2vec'] if m in MLP_GRID_CONFIG['input_method']]

# Preparar dados TF-IDF (silencioso)
tfidf_data = {}
if 'tfidf' in methods_to_run:
    for max_features in MLP_GRID_CONFIG['tfidf_params']['max_features']:
        for ngram_range in MLP_GRID_CONFIG['tfidf_params']['ngram_range']:
            key = f"tfidf_{max_features}_{ngram_range[0]}_{ngram_range[1]}"
            vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
            X_train_tfidf = vectorizer.fit_transform(X_train).toarray()
            X_test_tfidf = vectorizer.transform(X_test).toarray()
            tfidf_data[key] = {
                'X_train': X_train_tfidf,
                'X_test': X_test_tfidf,
                'vectorizer': vectorizer
            }

# Parâmetros do modelo
model_param_grid = list(ParameterGrid({
    'hidden_layers': MLP_GRID_CONFIG['model_params']['hidden_layers'],
    'dropout_rate': MLP_GRID_CONFIG['model_params']['dropout_rate'],
    'learning_rate': MLP_GRID_CONFIG['model_params']['learning_rate'],
    'batch_size': MLP_GRID_CONFIG['model_params']['batch_size']
}))

# Calcular total de combinações
total_combinations = 0
if 'tfidf' in methods_to_run:
    total_combinations += len(tfidf_data) * len(model_param_grid)
if 'word2vec' in methods_to_run:
    w2v_param_grid = list(ParameterGrid(MLP_GRID_CONFIG['word2vec_params']))
    total_combinations += len(w2v_param_grid) * len(model_param_grid)

# Iniciar grid-search
print("="*60)
print("INICIANDO GRID-SEARCH MLP")
print("="*60)

all_results = []
combination_count = 0
best_so_far = None
best_score_so_far = -1

# 1. TF-IDF (se configurado)
if 'tfidf' in methods_to_run:
    for tfidf_key, tfidf_data_dict in tfidf_data.items():
        X_train_prep = tfidf_data_dict['X_train']
        X_test_prep = tfidf_data_dict['X_test']
        vectorizer = tfidf_data_dict['vectorizer']
        
        for model_params in model_param_grid:
            combination_count += 1
            print(f"\n[{combination_count}/{total_combinations}] TF-IDF + Model")
            print(f"  Parâmetros:")
            print(f"    • TF-IDF: max_features={vectorizer.max_features}, ngram_range={vectorizer.ngram_range}")
            print(f"    • Hidden Layers: {model_params['hidden_layers']}")
            print(f"    • Dropout: {model_params['dropout_rate']}")
            print(f"    • Learning Rate: {model_params['learning_rate']}")
            print(f"    • Batch Size: {model_params['batch_size']}")
            
            try:
                cv_scores, test_score, model = train_and_evaluate_mlp(
                    X_train_prep, X_test_prep, y_train_oh, y_test_oh,
                    **model_params
                )
                
                result = {
                    'input_method': 'tfidf',
                    'tfidf_params': {
                        'max_features': vectorizer.max_features,
                        'ngram_range': vectorizer.ngram_range
                    },
                    'model_params': model_params,
                    'cv_mean': cv_scores.mean(),
                    'cv_std': cv_scores.std(),
                    'test_score': test_score,
                    'model': model,
                    'vectorizer': vectorizer
                }
                all_results.append(result)
                
                # Melhor modelo até agora
                if test_score > best_score_so_far:
                    best_score_so_far = test_score
                    best_so_far = result
                    print(f"  ⭐ Novo melhor! Test Score: {test_score:.4f}")
            except Exception as e:
                print(f"  ❌ Erro: {e}")

# 2. Word2Vec (se configurado)
if 'word2vec' in methods_to_run:
    w2v_param_grid = list(ParameterGrid(MLP_GRID_CONFIG['word2vec_params']))
    
    for w2v_params in w2v_param_grid:
        # Preparar dados Word2Vec (silencioso)
        try:
            X_train_w2v, X_test_w2v, w2v_model = prepare_data_with_word2vec(
                X_train, X_test,
                vector_size=w2v_params['vector_size'],
                window=w2v_params['window'],
                min_count=w2v_params['min_count'],
                aggregation=w2v_params['aggregation'],
                verbose=False
            )
            
            for model_params in model_param_grid:
                combination_count += 1
                print(f"\n[{combination_count}/{total_combinations}] Word2Vec + Model")
                print(f"  Parâmetros:")
                print(f"    • Word2Vec: vector_size={w2v_params['vector_size']}, window={w2v_params['window']}, min_count={w2v_params['min_count']}, aggregation={w2v_params['aggregation']}")
                print(f"    • Hidden Layers: {model_params['hidden_layers']}")
                print(f"    • Dropout: {model_params['dropout_rate']}")
                print(f"    • Learning Rate: {model_params['learning_rate']}")
                print(f"    • Batch Size: {model_params['batch_size']}")
                
                try:
                    cv_scores, test_score, model = train_and_evaluate_mlp(
                        X_train_w2v, X_test_w2v, y_train_oh, y_test_oh,
                        **model_params
                    )
                    
                    result = {
                        'input_method': 'word2vec',
                        'word2vec_params': w2v_params,
                        'model_params': model_params,
                        'cv_mean': cv_scores.mean(),
                        'cv_std': cv_scores.std(),
                        'test_score': test_score,
                        'model': model,
                        'w2v_model': w2v_model
                    }
                    all_results.append(result)
                    
                    # Melhor modelo até agora
                    if test_score > best_score_so_far:
                        best_score_so_far = test_score
                        best_so_far = result
                        print(f"  ⭐ Novo melhor! Test Score: {test_score:.4f}")
                except Exception as e:
                    print(f"  ❌ Erro: {e}")
        except Exception as e:
            print(f"  ❌ Erro ao preparar Word2Vec: {e}")

# ============================================================================
# RESULTADOS FINAIS
# ============================================================================

if all_results:
    # Melhor modelo geral (por test_score)
    best_overall = max(all_results, key=lambda x: x['test_score'])
    
    print("\n" + "="*60)
    print("MELHOR MODELO GERAL")
    print("="*60)
    print(f"Método: {best_overall['input_method'].upper()}")
    print(f"Test Score: {best_overall['test_score']:.4f}")
    print(f"CV Score: {best_overall['cv_mean']:.4f} ± {best_overall['cv_std']:.4f}")
    print(f"\n📋 Parâmetros do Melhor Modelo:")
    if best_overall['input_method'] == 'tfidf':
        print(f"  • TF-IDF:")
        print(f"    - max_features: {best_overall['tfidf_params']['max_features']}")
        print(f"    - ngram_range: {best_overall['tfidf_params']['ngram_range']}")
    else:
        print(f"  • Word2Vec:")
        print(f"    - vector_size: {best_overall['word2vec_params']['vector_size']}")
        print(f"    - window: {best_overall['word2vec_params']['window']}")
        print(f"    - min_count: {best_overall['word2vec_params']['min_count']}")
        print(f"    - aggregation: {best_overall['word2vec_params']['aggregation']}")
    print(f"  • MLP:")
    print(f"    - hidden_layers: {best_overall['model_params']['hidden_layers']}")
    print(f"    - dropout_rate: {best_overall['model_params']['dropout_rate']}")
    print(f"    - learning_rate: {best_overall['model_params']['learning_rate']}")
    print(f"    - batch_size: {best_overall['model_params']['batch_size']}")
    
    # Melhor TF-IDF
    if 'tfidf' in methods_to_run:
        tfidf_results = [r for r in all_results if r['input_method'] == 'tfidf']
        if tfidf_results:
            best_tfidf = max(tfidf_results, key=lambda x: x['test_score'])
            print("\n" + "="*60)
            print("MELHOR TF-IDF")
            print("="*60)
            print(f"Test Score: {best_tfidf['test_score']:.4f}")
            print(f"CV Score: {best_tfidf['cv_mean']:.4f} ± {best_tfidf['cv_std']:.4f}")
            print(f"\n📋 Parâmetros:")
            print(f"  • TF-IDF: max_features={best_tfidf['tfidf_params']['max_features']}, ngram_range={best_tfidf['tfidf_params']['ngram_range']}")
            print(f"  • MLP: hidden_layers={best_tfidf['model_params']['hidden_layers']}, dropout={best_tfidf['model_params']['dropout_rate']}, lr={best_tfidf['model_params']['learning_rate']}, batch={best_tfidf['model_params']['batch_size']}")
    
    # Melhor Word2Vec
    if 'word2vec' in methods_to_run:
        w2v_results = [r for r in all_results if r['input_method'] == 'word2vec']
        if w2v_results:
            best_w2v = max(w2v_results, key=lambda x: x['test_score'])
            print("\n" + "="*60)
            print("MELHOR WORD2VEC")
            print("="*60)
            print(f"Test Score: {best_w2v['test_score']:.4f}")
            print(f"CV Score: {best_w2v['cv_mean']:.4f} ± {best_w2v['cv_std']:.4f}")
            print(f"\n📋 Parâmetros:")
            print(f"  • Word2Vec: vector_size={best_w2v['word2vec_params']['vector_size']}, window={best_w2v['word2vec_params']['window']}, min_count={best_w2v['word2vec_params']['min_count']}, aggregation={best_w2v['word2vec_params']['aggregation']}")
            print(f"  • MLP: hidden_layers={best_w2v['model_params']['hidden_layers']}, dropout={best_w2v['model_params']['dropout_rate']}, lr={best_w2v['model_params']['learning_rate']}, batch={best_w2v['model_params']['batch_size']}")
    
    # Armazenar resultados
    mlp_results = {
        'model_name': f"MLP (Grid-Search - {best_overall['input_method'].upper()})",
        'best_result': best_overall,
        'all_results': all_results,
        'input_method': best_overall['input_method'],
        'best_params': {**best_overall.get('tfidf_params', {}), 
                        **best_overall.get('word2vec_params', {}), 
                        **best_overall['model_params']},
        'best_cv_score': best_overall['cv_mean'],
        'test_score': best_overall['test_score'],
        'model': best_overall['model']
    }
    
    if best_overall['input_method'] == 'tfidf':
        mlp_results['vectorizer'] = best_overall['vectorizer']
    else:
        mlp_results['w2v_model'] = best_overall['w2v_model']
    
    # Predições
    if best_overall['input_method'] == 'tfidf':
        tfidf_key = f"tfidf_{best_overall['tfidf_params']['max_features']}_{best_overall['tfidf_params']['ngram_range'][0]}_{best_overall['tfidf_params']['ngram_range'][1]}"
        X_test_prep = tfidf_data[tfidf_key]['X_test']
    else:
        X_train_w2v, X_test_prep, _ = prepare_data_with_word2vec(
            X_train, X_test,
            vector_size=best_overall['word2vec_params']['vector_size'],
            window=best_overall['word2vec_params']['window'],
            min_count=best_overall['word2vec_params']['min_count'],
            aggregation=best_overall['word2vec_params']['aggregation'],
            verbose=False
        )
    
    mlp_y_pred = best_overall['model'].predict(X_test_prep.astype('float32'))
    mlp_y_pred_labels = np.argmax(mlp_y_pred, axis=1) if mlp_y_pred.ndim > 1 else mlp_y_pred
    mlp_results['predictions'] = mlp_y_pred_labels
    
    print("\n" + "="*60)
else:
    print("\n❌ Nenhum resultado foi gerado!")
    mlp_results = {}



INICIANDO GRID-SEARCH MLP

[1/4] Word2Vec + Model
  Parâmetros:
    • Word2Vec: vector_size=300, window=5, min_count=2, aggregation=mean
    • Hidden Layers: (512, 256)
    • Dropout: 0.7
    • Learning Rate: 0.0001
    • Batch Size: 128
  📊 Fold 1/3 - Treinando...
Epoch 1/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3879 - loss: 2.0066 - learning_rate: 1.0000e-04
Epoch 2/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4343 - loss: 1.7130 - learning_rate: 1.0000e-04
Epoch 3/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4564 - loss: 1.5385 - learning_rate: 1.0000e-04
Epoch 4/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4662 - loss: 1.4476 - learning_rate: 1.0000e-04
Epoch 5/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.4788 - loss: 1.3362 - learning_rate: 1.0000e-04
Epoch 6/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4867 - loss: 1.2662 - learning_rate: 1.0000e-04
Epoch 7/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 2s 9m

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# CONFIGURAÇÃO BERT GRID-SEARCH
# ============================================================================

BERT_GRID_CONFIG = {
    'model_name': 'neuralmind/bert-base-portuguese-cased',  # BERTimbau
    'method': 'finetuning',
    'grid_params': {
        'learning_rate': [2e-5, 3e-5, 5e-5],
        'batch_size': [8, 16],
        'num_train_epochs': [2, 3],
        'weight_decay': [0.01, 0.1],
        'max_length': [256, 512]
    },
    'fixed_params': {
        'warmup_steps': 500,
        'evaluation_strategy': 'epoch',
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'accuracy'
    }
}

print("="*80)
print("CONFIGURAÇÃO BERT GRID-SEARCH")
print("="*80)
print(f"Model: {BERT_GRID_CONFIG['model_name']}")
print(f"Learning Rates: {BERT_GRID_CONFIG['grid_params']['learning_rate']}")
print(f"Batch Sizes: {BERT_GRID_CONFIG['grid_params']['batch_size']}")
print(f"Epochs: {BERT_GRID_CONFIG['grid_params']['num_train_epochs']}")
print(f"Weight Decay: {BERT_GRID_CONFIG['grid_params']['weight_decay']}")
print(f"Max Length: {BERT_GRID_CONFIG['grid_params']['max_length']}")
print(f"Total de combinações: {len(ParameterGrid(BERT_GRID_CONFIG['grid_params']))}")
print("="*80)


In [ ]:
# ============================================================================
# BERT COM GRID-SEARCH
# ============================================================================

# Carregar dados
X_train = dataset["X_train"]
X_test = dataset["X_test"]
y_train = dataset["y_train"]
y_test = dataset["y_test"]
label_encoder = dataset["label_encoder"]

print("="*80)
print("BERT COM GRID-SEARCH")
print("="*80)

# Função para preparar dataset
def prepare_bert_dataset(texts, labels, tokenizer, max_length=512):
    """Prepara dataset para BERT"""
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            padding='max_length',
            max_length=max_length
        )
    
    dataset_dict = {
        'text': texts.tolist() if isinstance(texts, np.ndarray) else texts,
        'labels': labels.tolist() if isinstance(labels, np.ndarray) else labels
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    
    return tokenized_dataset

# Função de métricas
def compute_metrics(eval_pred):
    """Calcula métricas para BERT (apenas acurácia)"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions)
    }

# Função para treinar BERT com parâmetros específicos
def train_bert_with_params(X_train, X_test, y_train, y_test, params):
    """Treina BERT com parâmetros específicos
    
    Fluxo correto:
    1. Separa X_train (85%) em X_train_internal (70%) + X_val_internal (15%)
    2. Treina usando train_dataset (70%) e eval_dataset (val_internal - 15%)
    3. Avalia no X_test (15% separado) apenas no final
    """
    from sklearn.model_selection import train_test_split
    
    print(f"\n{'='*60}")
    print(f"Testando: {params}")
    print(f"{'='*60}")
    
    # 1. SEPARAR TREINO EM TREINO INTERNO + VALIDAÇÃO INTERNA
    # X_train (85%) → X_train_internal (70%) + X_val_internal (15%)
    X_train_internal, X_val_internal, y_train_internal, y_val_internal = train_test_split(
        X_train, y_train,
        test_size=0.176,  # 15% de 85% ≈ 17.6% para obter ~15% do total
        stratify=y_train,
        random_state=42
    )
    
    print(f"  📊 Dados:")
    print(f"    • Treino interno: {len(X_train_internal):,} amostras (70%)")
    print(f"    • Validação interna: {len(X_val_internal):,} amostras (15%)")
    print(f"    • Teste: {len(X_test):,} amostras (15% - separado)")
    
    # Carregar modelo e tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BERT_GRID_CONFIG['model_name'])
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_GRID_CONFIG['model_name'],
        num_labels=3
    )
    
    # Preparar datasets
    train_dataset = prepare_bert_dataset(
        X_train_internal, y_train_internal, tokenizer, params['max_length']
    )
    val_dataset = prepare_bert_dataset(
        X_val_internal, y_val_internal, tokenizer, params['max_length']
    )
    test_dataset = prepare_bert_dataset(
        X_test, y_test, tokenizer, params['max_length']
    )
    
    # Training arguments
    output_dir = f'./bert_results_{hash(str(params))}'
    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=params['learning_rate'],
        per_device_train_batch_size=params['batch_size'],
        per_device_eval_batch_size=params['batch_size'],
        num_train_epochs=params['num_train_epochs'],
        weight_decay=params['weight_decay'],
        warmup_steps=BERT_GRID_CONFIG['fixed_params']['warmup_steps'],
        evaluation_strategy=BERT_GRID_CONFIG['fixed_params']['evaluation_strategy'],
        save_strategy=BERT_GRID_CONFIG['fixed_params']['save_strategy'],
        load_best_model_at_end=BERT_GRID_CONFIG['fixed_params']['load_best_model_at_end'],
        metric_for_best_model=BERT_GRID_CONFIG['fixed_params']['metric_for_best_model'],
        logging_dir=f'{output_dir}/logs',
        logging_steps=100,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),  # Usar mixed precision se GPU disponível
        report_to='none'  # Não reportar para wandb/tensorboard
    )
    
    # Trainer (usa val_dataset como eval_dataset durante treino)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,  # Validação interna (15%)
        compute_metrics=compute_metrics
    )
    
    # Treinar (avalia na validação interna durante treino)
    print(f"  🔄 Treinando (avaliando na validação interna)...")
    trainer.train()
    
    # Avaliar na validação interna (após treino)
    print(f"  🔄 Avaliando na validação interna...")
    val_results = trainer.evaluate()
    val_score = val_results['eval_accuracy']
    
    # Avaliar no teste (APENAS NO FINAL - não usado durante treino)
    print(f"  🔄 Avaliando no conjunto de teste (15% separado)...")
    test_results = trainer.evaluate(eval_dataset=test_dataset)
    test_score = test_results['eval_accuracy']
    
    print(f"  ✅ Val Score: {val_score:.4f} | Test Score: {test_score:.4f}")
    
    # Limpar diretório temporário
    import shutil
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    return {
        'params': params,
        'model': model,
        'tokenizer': tokenizer,
        'trainer': trainer,
        'val_results': val_results,
        'val_score': val_score,
        'test_results': test_results,
        'test_score': test_score
    }

# Grid-Search manual
print(f"\n{'='*80}")
print("INICIANDO GRID-SEARCH BERT")
print(f"{'='*80}")

param_grid = ParameterGrid(BERT_GRID_CONFIG['grid_params'])
bert_results = []
total_combinations = len(param_grid)

for idx, params in enumerate(param_grid, 1):
    print(f"\n{'='*80}")
    print(f"Combinação {idx}/{total_combinations}")
    print(f"{'='*80}")
    
    try:
        result = train_bert_with_params(X_train, X_test, y_train, y_test, params)
        bert_results.append(result)
        
        print(f"✅ Val Accuracy: {result['val_score']:.4f} | Test Accuracy: {result['test_score']:.4f}")
    except Exception as e:
        print(f"❌ Erro ao treinar: {e}")
        continue

# Melhor modelo
if bert_results:
    best_bert = max(bert_results, key=lambda x: x['test_score'])
    
    print(f"\n{'='*80}")
    print("RESULTADOS - BERT GRID-SEARCH")
    print(f"{'='*80}")
    print(f"Melhor modelo BERT (baseado em Test Score):")
    print(f"  Parâmetros: {best_bert['params']}")
    print(f"  Val Accuracy: {best_bert['val_score']:.4f}")
    print(f"  Test Accuracy: {best_bert['test_score']:.4f}")
    print(f"{'='*80}")
    
    # Armazenar resultados
    bert_final_results = {
        'model_name': 'BERT (Grid-Search)',
        'best_params': best_bert['params'],
        'test_score': best_bert['test_score'],
        'model': best_bert['model'],
        'tokenizer': best_bert['tokenizer'],
        'all_results': bert_results
    }
else:
    print(f"\n❌ Nenhum modelo BERT foi treinado com sucesso.")
    bert_final_results = None

print(f"\n{'='*80}")
print("✅ BERT GRID-SEARCH CONCLUÍDO")
print(f"{'='*80}")


In [ ]:
'# ============================================================================
# COMPARAÇÃO DE RESULTADOS
# ============================================================================

print("="*80)
print("COMPARAÇÃO DE RESULTADOS")
print("="*80)

# Preparar dados para comparação
comparison_data = {
    'Modelo': [],
    'Test Accuracy': []
}

# Adicionar coluna de CV Accuracy se disponível no baseline
if 'cv_mean' in baseline_results:
    comparison_data['CV Accuracy (10 folds)'] = []

# Baseline
comparison_data['Modelo'].append('Baseline (TF-IDF + Logistic Regression)')
comparison_data['Test Accuracy'].append(baseline_results['test_score'])
if 'cv_mean' in baseline_results:
    comparison_data['CV Accuracy (10 folds)'].append(
        f"{baseline_results['cv_mean']:.4f} ± {baseline_results['cv_std']:.4f}"
    )

# MLP
if 'mlp_results' in globals():
    comparison_data['Modelo'].append('MLP (Grid-Search)')
    comparison_data['Test Accuracy'].append(mlp_results['test_score'])
    if 'cv_mean' in baseline_results:
        comparison_data['CV Accuracy (10 folds)'].append(
            f"{mlp_results['best_cv_score']:.4f}"
        )

# BERT
if 'bert_final_results' in globals() and bert_final_results is not None:
    comparison_data['Modelo'].append('BERT (Grid-Search)')
    comparison_data['Test Accuracy'].append(bert_final_results['test_score'])
    if 'cv_mean' in baseline_results:
        comparison_data['CV Accuracy (10 folds)'].append('N/A')

# Criar DataFrame
comparison_df = pd.DataFrame(comparison_data)
print("\n📊 TABELA COMPARATIVA:")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualização
print(f"\n{'='*80}")
print("VISUALIZAÇÃO COMPARATIVA")
print(f"{'='*80}")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.35

bars = ax.bar(x, comparison_df['Test Accuracy'], width, label='Test Accuracy', color='steelblue')

ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Comparação de Modelos - Test Accuracy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Modelo'], rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for i, (bar, acc) in enumerate(zip(bars, comparison_df['Test Accuracy'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()

# Salvar figura
comparison_path = os.path.join(figures_dir, 'model_comparison.png')
plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
print(f"✅ Figura salva: {comparison_path}")
plt.show()

# Análise
print(f"\n{'='*80}")
print("ANÁLISE")
print(f"{'='*80}")

best_model_idx = comparison_df['Test Accuracy'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Modelo']
best_score = comparison_df.loc[best_model_idx, 'Test Accuracy']

print(f"🏆 Melhor Modelo: {best_model}")
print(f"   Test Accuracy: {best_score:.4f}")

print(f"\n📈 Melhoria em relação ao Baseline:")
baseline_acc = comparison_df.loc[0, 'Test Accuracy']
improvement = ((best_score - baseline_acc) / baseline_acc) * 100
print(f"   Baseline: {baseline_acc:.4f}")
print(f"   Melhor: {best_score:.4f}")
print(f"   Melhoria: {improvement:+.2f}%")

print(f"\n{'='*80}")
print("✅ COMPARAÇÃO CONCLUÍDA")
print(f"{'='*80}")
'